### ENTREGA

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import time
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json

In [ ]:
#Se definen las rutas de los accesorios.
carpeta = Path.cwd() / "accesorios"

ruta_sombrero = carpeta / "sombrero.png"
ruta_puro = carpeta / "puro.png"
ruta_bandana = carpeta / "bandadna.png"

#Se cargan las imágenes.
def cargar_accesorio(ruta):
    imagen = cv2.imread(str(ruta), cv2.IMREAD_UNCHANGED)

    if imagen is None:
        print(f"No se ha encontrado: {ruta.name}")
        return None

    #Se añade un canal alfa cuando la imagen no tiene transparencia.
    if len(imagen.shape) == 3 and imagen.shape[2] == 3:
        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2BGRA)

    return imagen

sombrero = cargar_accesorio(ruta_sombrero)
puro = cargar_accesorio(ruta_puro)
bandana = cargar_accesorio(ruta_bandana)

#Se recorta el espacio transparente del puro.
if puro is not None:
    mascara = puro[:, :, 3]
    puntos = cv2.findNonZero(mascara)
    x, y, w, h = cv2.boundingRect(puntos)
    puro = puro[y:y+h, x:x+w]

#Se define la función para la superposición de imágenes.
def superponer(imagen, accesorio, centro, ancho, angulo=0):

    if accesorio is None:
        return imagen

    alto_original, ancho_original = accesorio.shape[:2]

    #Se mantiene la proporcion original del accesorio.
    alto = int(alto_original * ancho / ancho_original)

    if ancho <= 0 or alto <= 0:
        return imagen

    accesorio = cv2.resize(accesorio, (ancho, alto))

    #Se gira el accesorio para adaptarlo a la inclinacion de la cabeza.
    matriz = cv2.getRotationMatrix2D((ancho // 2, alto // 2), -angulo, 1)

    accesorio = cv2.warpAffine(
        accesorio,
        matriz,
        (ancho, alto),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    alto_img, ancho_img = imagen.shape[:2]

    #Se calculan las coordenadas de la esquina superior izquierda.
    x = int(centro[0] - ancho / 2)
    y = int(centro[1] - alto / 2)

    #Se comprueba si el accesorio sobrepasa los limites de la imagen.
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(ancho_img, x + ancho)
    y2 = min(alto_img, y + alto)

    if x1 >= x2 or y1 >= y2:
        return imagen

    #Se recorta tambien la imagen del accesorio.
    accesorio_recortado = accesorio[y1 - y:y2 - y, x1 - x:x2 - x]

    #Se separan los canales de color y transparencia.
    zona = imagen[y1:y2, x1:x2]
    alfa = accesorio_recortado[:, :, 3] / 255.0
    alfa = alfa[:, :, np.newaxis]
    colores = accesorio_recortado[:, :, :3]

    #Se mezcla el accesorio con la imagen original.
    imagen[y1:y2, x1:x2] = (colores * alfa + zona * (1 - alfa)).astype(np.uint8)

    return imagen

#Se define una función para aplicar un filtro del oeste al vídeo.
def filtro_oeste(imagen):
    #Se convierte la imagen a float para realizar las operaciones.
    resultado = imagen.astype(np.float32)

    matriz_sepia = np.array([[0.272, 0.534, 0.131], [0.349, 0.686, 0.168], [0.393, 0.769, 0.189]])
    resultado = cv2.transform(resultado, matriz_sepia)
    resultado = np.clip(resultado, 0, 255)

    #Se establece el constraste y el brillo.
    resultado = cv2.convertScaleAbs(resultado, alpha=1.15, beta=5)

    #Se aplica el viñetado.
    alto, ancho = imagen.shape[:2]

    kernel_x = cv2.getGaussianKernel(ancho, ancho * 0.5)
    kernel_y = cv2.getGaussianKernel(alto, alto * 0.5)

    mascara = kernel_y @ kernel_x.T
    mascara = mascara / mascara.max()

    #Se oscurecen ligeramente las esquinas.
    mascara = 0.55 + 0.45 * mascara
    resultado = (resultado.astype(np.float32) * mascara[:, :, np.newaxis])

    #Se aplica grano cinematográfico.
    ruido = np.random.normal(0, 3, resultado.shape).astype(np.float32)
    resultado = resultado + ruido
    return np.clip(resultado, 0, 255).astype(np.uint8)

#Se define la detección facial
modelo = Path.cwd() / "face_landmarker.task"

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

opciones = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(modelo)),
    running_mode=RunningMode.VIDEO,
    num_faces=1
)

face_landmarker = FaceLandmarker.create_from_options(opciones)

#Se define la detección de manos.
modelo_manos = Path.cwd() / "hand_landmarker.task"

HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

#Se define la detección de pose.
modelo_pose = Path.cwd() / "pose_landmarker_lite.task"

PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions

opciones_pose = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(modelo_pose)),
    running_mode=RunningMode.VIDEO,
    num_poses=1
)

pose_landmarker = PoseLandmarker.create_from_options(opciones_pose)

opciones_manos = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(modelo_manos)),
    running_mode=RunningMode.VIDEO,
    num_hands=2
)

hand_landmarker = HandLandmarker.create_from_options(opciones_manos)

#Se hace la captura de vídeo.
camara = cv2.VideoCapture(0)

if not camara.isOpened():
    print("No se ha podido acceder a la camara.")
    exit()

ultimo_timestamp = -1
estado_puro = "boca"
tiempo_cerca = None
mano_alejada = False
tiempo_recogida = 0.9
tiempo_devolucion = 0.3

while True:

    correcto, frame = camara.read()

    if not correcto:
        break

    #Se refleja la imagen para simular un espejo.
    frame = cv2.flip(frame, 1)
    alto, ancho = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    #Se convierte la imagen al formato utilizado por MediaPipe.
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    #Se obtiene una marca de tiempo creciente.
    timestamp_ms = max(int(time.monotonic() * 1000), ultimo_timestamp + 1)
    ultimo_timestamp = timestamp_ms

    #Se detectan los puntos faciales.
    resultados = face_landmarker.detect_for_video(mp_image, timestamp_ms)

    #Se detectan los puntos de las manos.
    resultados_manos = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

    #Se detectan los puntos del cuerpo.
    resultados_pose = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

    if resultados.face_landmarks:
        cara = resultados.face_landmarks[0]

        #Se obtienen las coordenadas de un punto facial.
        def punto(indice):
            landmark = cara[indice]

            x = int(landmark.x * ancho)
            y = int(landmark.y * alto)

            return (x, y)

        #Se obtienen los puntos principales de la cara.
        frente = punto(6)
        lado_izquierdo = punto(234)
        lado_derecho = punto(454)
        comisura_derecha = punto(291)

        #Se calcula la anchura aproximada de la cara.
        ancho_cara = int(np.linalg.norm(np.array(lado_derecho) - np.array(lado_izquierdo)))

        #Se calcula la inclinacion de la cabeza.
        vector_cara = (np.array(lado_derecho) - np.array(lado_izquierdo))
        angulo = np.degrees(np.arctan2(vector_cara[1], vector_cara[0]))

        #Se establece el sombrero en el vídeo.
        if sombrero is not None:
            ancho_sombrero = int(ancho_cara * 2.2)
            alto_sombrero = int(sombrero.shape[0] * ancho_sombrero / sombrero.shape[1])
            centro_sombrero = (frente[0], frente[1] - int(alto_sombrero * 0.45))
            frame = superponer(frame, sombrero, centro_sombrero, ancho_sombrero, angulo)

        #Se establece la bandana en el vídeo.
        if resultados_pose.pose_landmarks and bandana is not None:
            pose = resultados_pose.pose_landmarks[0]

            #Se obtienen las posiciones de los hombros.
            hombro_izquierdo = (int(pose[11].x * ancho), int(pose[11].y * alto))
            hombro_derecho = (int(pose[12].x * ancho), int(pose[12].y * alto))

            #Se calcula la distancia entre los hombros.
            distancia_hombros = int(
                np.linalg.norm(np.array(hombro_derecho) - np.array(hombro_izquierdo))
            )

            #Se calcula el centro entre ambos hombros.
            centro_hombros = (
                int((hombro_izquierdo[0] + hombro_derecho[0]) / 2),
                int((hombro_izquierdo[1] + hombro_derecho[1]) / 2)
            )

            #Se desplaza la bandana ligeramente hacia abajo.
            centro_bandana = (centro_hombros[0], centro_hombros[1] + int(distancia_hombros * 0.06))
            ancho_bandana = int(distancia_hombros * 1.10)

            #Se calcula la inclinacion de los hombros.
            vector_hombros = (np.array(hombro_derecho) - np.array(hombro_izquierdo))
            angulo_bandana = np.degrees(np.arctan2(vector_hombros[1], -vector_hombros[0]))

            frame = superponer(frame, bandana, centro_bandana, ancho_bandana, -angulo_bandana)

        #Se establece el puro en el vídeo.
        if puro is not None:
            #Se calcula el tamaño del puro respecto a la cara.
            ancho_puro = int(ancho_cara * 0.45)
            alto_puro = int(puro.shape[0] * ancho_puro / puro.shape[1])

            #Se define el punto de anclaje en el extremo apagado del puro.
            ancla_x = int(ancho_puro * 0.96)
            ancla_y = int(alto_puro * 0.04)

            #Se utiliza la comisura como punto de sujecion del puro.
            destino = comisura_derecha

            #Se calcula el centro de la imagen para colocar
            #el extremo apagado exactamente en la comisura.
            centro_puro_boca = (
                destino[0] - ancla_x + ancho_puro,
                destino[1] - ancla_y + alto_puro // 2
            )

            #Se detecta la posición de la mano
            punto_mano = None

            if resultados_manos.hand_landmarks:
                distancia_minima = float("inf")

                for mano in resultados_manos.hand_landmarks:
                    #Se obtienen las puntas del pulgar y del indice.
                    pulgar = mano[4]
                    indice = mano[8]

                    #Se calcula el punto medio entre ambos dedos.
                    x_mano = int((pulgar.x + indice.x) / 2 * ancho)
                    y_mano = int((pulgar.y + indice.y) / 2 * alto)
                    candidato = (x_mano, y_mano)

                    #Se busca la mano mas cercana al puro.
                    distancia = np.linalg.norm(np.array(candidato) - np.array(centro_puro_boca))

                    if distancia < distancia_minima:
                        distancia_minima = distancia
                        punto_mano = candidato

            #Se calcula la distancia entre la mano y el puro en la boca.
            if punto_mano is not None:
                distancia_boca = np.linalg.norm(np.array(punto_mano) - np.array(centro_puro_boca))
            else:
                distancia_boca = float("inf")

            tiempo_actual = time.monotonic()
            umbral = int(ancho_cara * 0.20)

            if estado_puro == "boca":

                if distancia_boca < umbral:
                    if tiempo_cerca is None:
                        tiempo_cerca = tiempo_actual
                    elif tiempo_actual - tiempo_cerca >= tiempo_recogida:
                        estado_puro = "mano"
                        mano_alejada = False
                        tiempo_cerca = None
                else:
                    tiempo_cerca = None

                #Se muestra el puro en su posicion original.
                if estado_puro == "boca":
                    frame = superponer(frame, puro, centro_puro_boca, ancho_puro, 0)

            elif estado_puro == "mano":
                if punto_mano is not None:

                    #Se espera a que la mano se aleje de la boca.
                    if distancia_boca > umbral * 1.8:
                        mano_alejada = True
                    
                    #Se comprueba si la mano vuelve a acercarse a la boca.
                    if mano_alejada and distancia_boca < umbral:
                        if tiempo_cerca is None:
                            tiempo_cerca = tiempo_actual
                        elif tiempo_actual - tiempo_cerca >= tiempo_devolucion:
                            estado_puro = "boca"
                            mano_alejada = False
                            tiempo_cerca = None
                    else:
                        tiempo_cerca = None

                    #Se muestra el puro en la mano.
                    if estado_puro == "mano":
                        centro_puro_mano = (punto_mano[0] + int(ancho_puro * 0.20), punto_mano[1])
                        frame = superponer(frame, puro, centro_puro_mano, ancho_puro, 0)
                else:
                    tiempo_cerca = None

    #Se aplica el filtro del Oeste.
    frame = filtro_oeste(frame)

    #Se muestra el resultado.
    cv2.imshow("Filtro de vaquero", frame)

    #Se permite salir pulsando ESC.
    if cv2.waitKey(1) & 0xFF == 27:
        break

#Se liberan los recursos.
camara.release()
face_landmarker.close()
pose_landmarker.close()
cv2.destroyAllWindows()

descarga, separacion y entrenamiento

pip install -q kagglehub torch torchvision

In [ ]:
import kagglehub
from pathlib import Path

# descargar el dataset
ruta_descarga = Path(kagglehub.dataset_download("ananthu017/emotion-detection-fer"))

print("Dataset descargado en:", ruta_descarga)

# localizar las carpetas train y test
carpeta_train = None
carpeta_test = None

for ruta in ruta_descarga.rglob("*"):
    if ruta.is_dir():
        if ruta.name.lower() == "train":
            carpeta_train = ruta
        elif ruta.name.lower() == "test":
            carpeta_test = ruta

if carpeta_train is None or carpeta_test is None:
    raise FileNotFoundError("No se han encontrado las carpetas train y test.")

print("Carpeta de entrenamiento:", carpeta_train)
print("Carpeta de test:", carpeta_test)

# comprobar las clases disponibles
clases_train = sorted([c.name for c in carpeta_train.iterdir() if c.is_dir()])
clases_test = sorted([c.name for c in carpeta_test.iterdir() if c.is_dir()])

print("\nClases de entrenamiento:", clases_train)
print("Clases de test:", clases_test)

if clases_train != clases_test:
    raise ValueError("Las clases de train y test no coinciden.")

In [ ]:
import random
import shutil
from collections import Counter

# fijar semilla para reproducir la separacion
random.seed(42)

# carpeta donde guardaremos el dataset preparado
ruta_dataset = Path.cwd() / "dataset_emociones"

# evitar reconstruir accidentalmente una separacion anterior
if ruta_dataset.exists():
    raise FileExistsError(
        f"Ya existe {ruta_dataset}. No se ha modificado. "
        "Eliminala manualmente si quieres generar una nueva separacion."
    )

# crear carpetas principales
for conjunto in ["train", "val", "test"]:
    for clase in clases_train:
        (ruta_dataset / conjunto / clase).mkdir(parents=True, exist_ok=True)

# separar las imagenes originales de entrenamiento
for clase in clases_train:

    imagenes = [
        f for f in (carpeta_train / clase).iterdir()
        if f.is_file()
    ]

    random.shuffle(imagenes)

    limite = int(len(imagenes) * 0.8)

    imagenes_train = imagenes[:limite]
    imagenes_val = imagenes[limite:]

    # copiar imagenes de entrenamiento
    for imagen in imagenes_train:
        shutil.copy2(imagen, ruta_dataset / "train" / clase / imagen.name)

    # copiar imagenes de validacion
    for imagen in imagenes_val:
        shutil.copy2(imagen, ruta_dataset / "val" / clase / imagen.name)

    print(f"{clase}: {len(imagenes_train)} train, {len(imagenes_val)} val")

# copiar el conjunto de test original sin dividirlo
for clase in clases_test:

    imagenes = [
        f for f in (carpeta_test / clase).iterdir()
        if f.is_file()
    ]

    for imagen in imagenes:
        shutil.copy2(imagen, ruta_dataset / "test" / clase / imagen.name)

print("\nSeparacion completada.")

# mostrar distribucion final
for conjunto in ["train", "val", "test"]:
    print(f"\n{conjunto.upper()}")

    for clase in clases_train:
        cantidad = len(list((ruta_dataset / conjunto / clase).glob("*")))
        print(f"{clase}: {cantidad}")

In [ ]:

# --------------------------------------------------
# IMPORTACIONES Y CONFIGURACION
# --------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

import numpy as np
import matplotlib.pyplot as plt
import json
import copy
import random

from pathlib import Path
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


# fijar semillas para reproducir los resultados
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# seleccionar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# --------------------------------------------------
# CONFIGURACION DEL ENTRENAMIENTO
# --------------------------------------------------

IMG_SIZE = 224
BATCH_SIZE = 32

EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 40

LR_HEAD = 0.001
LR_BACKBONE = 0.00001
LR_FINETUNE_HEAD = 0.0001

PATIENCE = 10

# directorios del dataset
ruta_dataset = Path.cwd() / "dataset_emociones"

carpeta_train = ruta_dataset / "train"
carpeta_val = ruta_dataset / "val"
carpeta_test = ruta_dataset / "test"

if not carpeta_val.exists():
    carpeta_val = ruta_dataset / "validation"

if not carpeta_val.exists():
    raise FileNotFoundError("No se ha encontrado la carpeta val o validation.")

for carpeta in [carpeta_train, carpeta_val, carpeta_test]:
    if not carpeta.exists():
        raise FileNotFoundError(f"No se ha encontrado: {carpeta}")


# --------------------------------------------------
# TRANSFORMACIONES
# --------------------------------------------------

# normalizacion utilizada por ResNet18 preentrenada en ImageNet
media = [0.485, 0.456, 0.406]
desviacion = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(media, desviacion)
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(media, desviacion)
])


# --------------------------------------------------
# CARGA DEL DATASET
# --------------------------------------------------

train_dataset = ImageFolder(carpeta_train, transform=transform_train)
val_dataset = ImageFolder(carpeta_val, transform=transform_eval)
test_dataset = ImageFolder(carpeta_test, transform=transform_eval)

clases = train_dataset.classes
num_clases = len(clases)

if num_clases != 7:
    raise ValueError(f"Se esperaban 7 clases, pero se encontraron {num_clases}.")

if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise ValueError("Las clases de train y val no coinciden.")

if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("Las clases de train y test no coinciden.")

print("\nClases:", clases)
print("Imagenes de entrenamiento:", len(train_dataset))
print("Imagenes de validacion:", len(val_dataset))
print("Imagenes de test:", len(test_dataset))


# crear los DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# --------------------------------------------------
# PESOS PARA EQUILIBRAR LAS CLASES
# --------------------------------------------------

# calcular cuantas imagenes hay de cada clase
conteo_clases = np.bincount(train_dataset.targets, minlength=num_clases)

# dar mas peso a las clases con menos imagenes
pesos_clases = len(train_dataset) / (num_clases * conteo_clases)

pesos_clases = torch.tensor(
    pesos_clases,
    dtype=torch.float32
).to(device)

print("\nDistribucion de entrenamiento:")
for clase, cantidad, peso in zip(clases, conteo_clases, pesos_clases):
    print(f"{clase}: {cantidad} imagenes | Peso: {peso.item():.3f}")


# --------------------------------------------------
# CREAR RESNET18 PREENTRENADA
# --------------------------------------------------

# cargar los pesos aprendidos con ImageNet
pesos = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=pesos)

# congelar inicialmente todas las capas
for param in model.parameters():
    param.requires_grad = False

# sustituir la capa clasificadora original
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, num_clases)
)

model = model.to(device)

print("\nResNet18 preparada.")
print("Numero de emociones:", num_clases)


# --------------------------------------------------
# FUNCIONES DE ENTRENAMIENTO Y VALIDACION
# --------------------------------------------------

criterion_train = nn.CrossEntropyLoss(weight=pesos_clases)

# validacion sin ponderar para medir el rendimiento general
criterion_eval = nn.CrossEntropyLoss()


def ejecutar_epoca(loader, entrenamiento=False):

    if entrenamiento:
        model.train()
    else:
        model.eval()

    # en la primera fase mantenemos el backbone en modo evaluacion
    if entrenamiento and fase_actual == "transfer":
        model.conv1.eval()
        model.bn1.eval()
        model.layer1.eval()
        model.layer2.eval()
        model.layer3.eval()
        model.layer4.eval()

    perdida_total = 0.0
    aciertos = 0
    total = 0

    for imagenes, etiquetas in loader:

        imagenes = imagenes.to(device)
        etiquetas = etiquetas.to(device)

        if entrenamiento:
            optimizer.zero_grad()

            salidas = model(imagenes)
            perdida = criterion_train(salidas, etiquetas)

            perdida.backward()
            optimizer.step()

        else:
            with torch.no_grad():
                salidas = model(imagenes)
                perdida = criterion_eval(salidas, etiquetas)

        predicciones = torch.argmax(salidas, dim=1)

        perdida_total += perdida.item() * imagenes.size(0)
        aciertos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

    perdida_media = perdida_total / total
    accuracy = aciertos / total

    return perdida_media, accuracy


# --------------------------------------------------
# PREPARAR CARPETA PARA GUARDAR RESULTADOS
# --------------------------------------------------

# crear una carpeta nueva sin sobrescribir experimentos anteriores
carpeta_modelos = Path.cwd() / "models"
carpeta_modelos.mkdir(exist_ok=True)

numero_experimento = 1

while (carpeta_modelos / f"resnet18_experimento_{numero_experimento}").exists():
    numero_experimento += 1

carpeta_resultados = carpeta_modelos / f"resnet18_experimento_{numero_experimento}"
carpeta_resultados.mkdir()

ruta_mejor_modelo = carpeta_resultados / "mejor_modelo.pth"
ruta_clases = carpeta_resultados / "clases.json"

print("\nResultados en:", carpeta_resultados)


# --------------------------------------------------
# VARIABLES PARA REGISTRAR EL ENTRENAMIENTO
# --------------------------------------------------

historial = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

mejor_val_loss = float("inf")
mejor_estado = None

fase_actual = "transfer"


# --------------------------------------------------
# FUNCION PARA ENTRENAR CADA FASE
# --------------------------------------------------

def entrenar_fase(nombre, epochs, optimizer, scheduler):

    global mejor_val_loss, mejor_estado, fase_actual

    fase_actual = nombre

    contador_sin_mejora = 0

    print("\n" + "=" * 55)
    print("INICIANDO FASE:", nombre.upper())
    print("=" * 55)

    for epoch in range(epochs):

        perdida_train, accuracy_train = ejecutar_epoca(
            train_loader,
            entrenamiento=True
        )

        perdida_val, accuracy_val = ejecutar_epoca(
            val_loader,
            entrenamiento=False
        )

        historial["train_loss"].append(perdida_train)
        historial["val_loss"].append(perdida_val)
        historial["train_acc"].append(accuracy_train)
        historial["val_acc"].append(accuracy_val)

        # actualizar el learning rate
        scheduler.step(perdida_val)

        print(
            f"Epoca {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {perdida_train:.4f} | "
            f"Train Acc: {accuracy_train:.4f} | "
            f"Val Loss: {perdida_val:.4f} | "
            f"Val Acc: {accuracy_val:.4f}"
        )

        # guardar el mejor modelo global
        if perdida_val < mejor_val_loss:

            mejor_val_loss = perdida_val

            mejor_estado = copy.deepcopy(model.state_dict())

            torch.save(mejor_estado, ruta_mejor_modelo)

            contador_sin_mejora = 0

            print("  Mejor modelo guardado.")

        else:
            contador_sin_mejora += 1

        # early stopping
        if contador_sin_mejora >= PATIENCE:

            print("Early Stopping activado.")
            break


# --------------------------------------------------
# FASE 1: TRANSFER LEARNING
# --------------------------------------------------

# solo se entrenara la cabeza clasificadora
optimizer = optim.AdamW(
    model.fc.parameters(),
    lr=LR_HEAD,
    weight_decay=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=0.000001
)

entrenar_fase(
    nombre="transfer",
    epochs=EPOCHS_HEAD,
    optimizer=optimizer,
    scheduler=scheduler
)


# --------------------------------------------------
# FASE 2: FINE-TUNING
# --------------------------------------------------

print("\nDescongelando toda la ResNet18...")

# permitir que todas las capas aprendan
for param in model.parameters():
    param.requires_grad = True

# usar una tasa menor para el backbone
optimizer = optim.AdamW(
    [
        {
            "params": [
                p for name, p in model.named_parameters()
                if not name.startswith("fc.")
            ],
            "lr": LR_BACKBONE
        },
        {
            "params": model.fc.parameters(),
            "lr": LR_FINETUNE_HEAD
        }
    ],
    weight_decay=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=4,
    min_lr=0.000001
)

entrenar_fase(
    nombre="finetuning",
    epochs=EPOCHS_FINETUNE,
    optimizer=optimizer,
    scheduler=scheduler
)


# --------------------------------------------------
# RECUPERAR EL MEJOR MODELO
# --------------------------------------------------

model.load_state_dict(mejor_estado)
model.eval()

print("\nMejor perdida de validacion:", round(mejor_val_loss, 4))


# --------------------------------------------------
# EVALUACION FINAL CON TEST
# --------------------------------------------------

y_real = []
y_pred = []

perdida_test = 0.0
total_test = 0

with torch.no_grad():

    for imagenes, etiquetas in test_loader:

        imagenes = imagenes.to(device)
        etiquetas = etiquetas.to(device)

        salidas = model(imagenes)

        perdida = criterion_eval(salidas, etiquetas)

        predicciones = torch.argmax(salidas, dim=1)

        perdida_test += perdida.item() * imagenes.size(0)
        total_test += etiquetas.size(0)

        y_real.extend(etiquetas.cpu().numpy())
        y_pred.extend(predicciones.cpu().numpy())


perdida_test = perdida_test / total_test
accuracy_test = np.mean(np.array(y_real) == np.array(y_pred))

print("\n" + "=" * 55)
print("RESULTADOS FINALES")
print("=" * 55)

print(f"Test Loss: {perdida_test:.4f}")
print(f"Test Accuracy: {accuracy_test:.4f}")

print("\nInforme de clasificacion:\n")

print(
    classification_report(
        y_real,
        y_pred,
        labels=list(range(num_clases)),
        target_names=clases,
        zero_division=0
    )
)


# --------------------------------------------------
# MATRIZ DE CONFUSION
# --------------------------------------------------

matriz = confusion_matrix(
    y_real,
    y_pred,
    labels=list(range(num_clases))
)

fig, ax = plt.subplots(figsize=(9, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=clases
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45
)

plt.title("Matriz de confusion - ResNet18")
plt.tight_layout()

plt.savefig(carpeta_resultados / "matriz_confusion.png")
plt.show()


# --------------------------------------------------
# GRAFICAS DE ENTRENAMIENTO
# --------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# accuracy
axes[0].plot(historial["train_acc"], label="Entrenamiento")
axes[0].plot(historial["val_acc"], label="Validacion")
axes[0].set_title("Precision del modelo")
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

# loss
axes[1].plot(historial["train_loss"], label="Entrenamiento")
axes[1].plot(historial["val_loss"], label="Validacion")
axes[1].set_title("Perdida del modelo")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

plt.savefig(carpeta_resultados / "graficas_entrenamiento.png")
plt.show()


# --------------------------------------------------
# GUARDAR CLASES Y RESULTADOS
# --------------------------------------------------

with open(ruta_clases, "w", encoding="utf-8") as archivo:
    json.dump(clases, archivo, ensure_ascii=False, indent=4)

# guardar tambien los resultados principales
resultados = {
    "test_loss": float(perdida_test),
    "test_accuracy": float(accuracy_test),
    "best_val_loss": float(mejor_val_loss),
    "epocas_ejecutadas": len(historial["train_loss"]),
    "clases": clases
}

with open(carpeta_resultados / "resultados.json", "w", encoding="utf-8") as archivo:
    json.dump(resultados, archivo, ensure_ascii=False, indent=4)


print("\n" + "=" * 55)
print("ENTRENAMIENTO FINALIZADO")
print("=" * 55)

print("Modelo:", ruta_mejor_modelo)
print("Clases:", ruta_clases)
print("Resultados:", carpeta_resultados)


![Resultados del entrenamiento](models/resnet18_reanudacion_1/resultados_entrenamiento.png)
![Gráfica de precisión](models/resnet18_reanudacion_1/graficas_entrenamiento.png)
![Matriz de confusión](models/resnet18_reanudacion_1/matriz_confusion.png)

In [ ]:
# --------------------------------------------------
# DETECCION DE MANOS
# --------------------------------------------------

modelo_manos = Path.cwd() / "hand_landmarker.task"

BaseOptions = mp.tasks.BaseOptions
RunningMode = mp.tasks.vision.RunningMode

HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

opciones_manos = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(modelo_manos)
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=2
)

hand_landmarker = HandLandmarker.create_from_options(
    opciones_manos
)


# --------------------------------------------------
# FUNCION PARA DIBUJAR LA EMPUÑADURA
# --------------------------------------------------


def dibujar_empunadura(imagen, centro, direccion, tamano):

    # Comprobar que tenemos una dirección válida.
    magnitud = np.linalg.norm(direccion)

    if magnitud < 0.05:

        direccion = np.array([
            np.cos(np.radians(ultimo_angulo)),
            np.sin(np.radians(ultimo_angulo))
        ])

    # Evitar tamaños demasiado pequeños.
    tamano = max(2.0, float(tamano))

    # Calcular un grosor válido para OpenCV.
    def grosor(factor):
        return max(1, int(tamano * factor))

    # Vector perpendicular para calcular el grosor.
    perpendicular = np.array([
        -direccion[1],
        direccion[0]
    ])

    magnitud_perp = np.linalg.norm(perpendicular)

    if magnitud_perp > 0:
        perpendicular = perpendicular / magnitud_perp

    # Función auxiliar para calcular puntos sobre la empuñadura.
    def punto(posicion, desplazamiento=0):

        return (
            int(
                centro[0] +
                direccion[0] * tamano * posicion +
                perpendicular[0] * tamano * desplazamiento
            ),
            int(
                centro[1] +
                direccion[1] * tamano * posicion +
                perpendicular[1] * tamano * desplazamiento
            )
        )

    # Extremos de la empuñadura.
    parte_trasera = punto(-1.5)
    parte_delantera = punto(0.9)

    # Mango exterior.
    cv2.line(
        imagen,
        parte_trasera,
        parte_delantera,
        (25, 25, 30),
        grosor(0.65),
        cv2.LINE_AA
    )

    # Cuerpo metálico interior.
    cv2.line(
        imagen,
        punto(-1.4),
        punto(0.75),
        (130, 135, 145),
        grosor(0.42),
        cv2.LINE_AA
    )

    # Zona de agarre.
    cv2.line(
        imagen,
        punto(-1.15),
        punto(-0.15),
        (45, 45, 50),
        grosor(0.48),
        cv2.LINE_AA
    )

    # Detalles de la zona de agarre.
    for posicion in [-1.0, -0.7, -0.4]:

        cv2.line(
            imagen,
            punto(posicion, -0.22),
            punto(posicion, 0.22),
            (105, 105, 115),
            grosor(0.045),
            cv2.LINE_AA
        )

    # Anillo metálico trasero.
    cv2.line(
        imagen,
        punto(-1.45),
        punto(-1.35),
        (190, 195, 205),
        grosor(0.72),
        cv2.LINE_AA
    )

    # Anillo metálico delantero.
    cv2.line(
        imagen,
        punto(0.65),
        punto(0.85),
        (190, 195, 205),
        grosor(0.75),
        cv2.LINE_AA
    )

    # Emisor de la espada.
    cv2.line(
        imagen,
        punto(0.85),
        punto(1.0),
        (75, 80, 90),
        grosor(0.95),
        cv2.LINE_AA
    )

    # Pequeño detalle luminoso del emisor.
    cv2.line(
        imagen,
        punto(0.95),
        punto(1.0),
        (180, 200, 220),
        grosor(0.18),
        cv2.LINE_AA
    )

    # Botón de activación.
    boton = punto(-0.1, -0.28)

    cv2.circle(
        imagen,
        boton,
        max(2, int(tamano * 0.10)),
        (30, 40, 210),
        -1,
        cv2.LINE_AA
    )

    return imagen

# --------------------------------------------------
# FUNCION PARA SUPERPONER LA ESPADA
# --------------------------------------------------

def superponer_espada(imagen, accesorio, centro, ancho, angulo):

    if accesorio is None:
        return imagen

    alto_original, ancho_original = accesorio.shape[:2]

    alto = int(alto_original * ancho / ancho_original)

    accesorio = cv2.resize(accesorio, (ancho, alto))

    # Rotar la imagen conservando todo su contenido.
    matriz = cv2.getRotationMatrix2D(
        (ancho // 2, alto // 2),
        -angulo,
        1
    )

    coseno = abs(matriz[0, 0])
    seno = abs(matriz[0, 1])

    nuevo_ancho = int(alto * seno + ancho * coseno)
    nuevo_alto = int(alto * coseno + ancho * seno)

    matriz[0, 2] += nuevo_ancho / 2 - ancho / 2
    matriz[1, 2] += nuevo_alto / 2 - alto / 2

    accesorio = cv2.warpAffine(
        accesorio,
        matriz,
        (nuevo_ancho, nuevo_alto),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    alto_img, ancho_img = imagen.shape[:2]

    x = int(centro[0] - nuevo_ancho / 2)
    y = int(centro[1] - nuevo_alto / 2)

    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(ancho_img, x + nuevo_ancho)
    y2 = min(alto_img, y + nuevo_alto)

    if x1 >= x2 or y1 >= y2:
        return imagen

    accesorio_recortado = accesorio[
        y1-y:y2-y,
        x1-x:x2-x
    ]

    zona = imagen[y1:y2, x1:x2]

    alfa = accesorio_recortado[:, :, 3] / 255.0
    alfa = alfa[:, :, np.newaxis]

    colores = accesorio_recortado[:, :, :3]

    imagen[y1:y2, x1:x2] = (
        colores * alfa + zona * (1-alfa)
    ).astype(np.uint8)

    return imagen


# --------------------------------------------------
# FUNCION PARA DIBUJAR LA HOJA LASER
# --------------------------------------------------

def dibujar_hoja(imagen, inicio, fin, color):

    # Crear una capa para el resplandor.
    brillo = np.zeros_like(imagen)

    # Dibujar la parte exterior de la hoja.
    cv2.line(
        brillo,
        inicio,
        fin,
        color,
        25
    )

    # Difuminar para conseguir el efecto luminoso.
    brillo = cv2.GaussianBlur(
        brillo,
        (31, 31),
        0
    )

    imagen = cv2.addWeighted(
        imagen,
        1,
        brillo,
        0.8,
        0
    )

    # Dibujar el núcleo de la hoja.
    cv2.line(
        imagen,
        inicio,
        fin,
        color,
        10
    )

    # Núcleo más claro.
    cv2.line(
        imagen,
        inicio,
        fin,
        (255, 255, 255),
        3
    )

    return imagen


# --------------------------------------------------
# VARIABLES DE CONTROL
# --------------------------------------------------

ultima_posicion_mano = None

ultimo_timestamp = -1
ultimo_angulo = 0

# Ajuste de orientación de la imagen.
angulo_offset = 0

# --------------------------------------------------
# CARGAR MODELO DE EMOCIONES
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargar las clases guardadas durante el entrenamiento.
with open(r"/models/resnet18_reanudacion_1/clases.json", "r", encoding="utf-8") as archivo:
    clases = json.load(archivo)

# Crear la misma arquitectura utilizada durante el entrenamiento.
modelo = models.resnet18(weights=None)

modelo.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(modelo.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, len(clases))
)

# Cargar los pesos entrenados.
modelo.load_state_dict(torch.load(
    r"/models/resnet18_reanudacion_1/modelo_resnet18_final.pth",
    map_location=device
))

modelo = modelo.to(device)
modelo.eval()

# Mismas transformaciones que durante la evaluación.
transformacion = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

print("Modelo de emociones cargado.")
print("Clases:", clases)

# --------------------------------------------------
# DETECCION DE CARAS
# --------------------------------------------------

detector_caras = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

emocion_actual = None
color_sable = (255, 0, 0)  # Azul inicial (BGR)

ultima_prediccion = 0

# --------------------------------------------------
# CLASIFICAR EMOCION
# --------------------------------------------------

def detectar_emocion(frame):

    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    caras = detector_caras.detectMultiScale(
        gris,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    if len(caras) == 0:
        return None

    # Seleccionar la cara de mayor tamaño.
    x, y, w, h = max(caras, key=lambda cara: cara[2] * cara[3])

    recorte = frame[y:y+h, x:x+w]

    # Convertir a escala de grises.
    recorte = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)

    # Volver a convertir a RGB para mantener los tres canales.
    recorte = cv2.cvtColor(recorte, cv2.COLOR_GRAY2RGB)

    # Aplicar las mismas transformaciones del entrenamiento.
    imagen = Image.fromarray(recorte)
    imagen = transformacion(imagen).unsqueeze(0).to(device)

    with torch.no_grad():
        salida = modelo(imagen)
        prediccion = torch.argmax(salida, dim=1).item()

    return clases[prediccion]

# --------------------------------------------------
# COLORES SEGUN LA EMOCION
# --------------------------------------------------

colores_emociones = {
    "angry": (0, 0, 255),       # Rojo
    "happy": (0, 255, 0),       # Verde
    "sad": (255, 0, 0),         # Azul
    "neutral": (255, 0, 255)    # Morado
}

# --------------------------------------------------
# CAPTURA DE VIDEO
# --------------------------------------------------

camara = cv2.VideoCapture(0)

if not camara.isOpened():
    print("No se ha podido acceder a la cámara.")

else:

    while True:

        correcto, frame = camara.read()

        if not correcto:
            break

        # Reflejar la imagen como si fuera un espejo.
        frame = cv2.flip(frame, 1)

        # --------------------------------------------------
        # ACTUALIZAR COLOR DEL SABLE
        # --------------------------------------------------

        tiempo_actual = time.monotonic()

        # Actualizar la emocion cada 0.5 segundos.
        if tiempo_actual - ultima_prediccion >= 0.5:

            emocion = detectar_emocion(frame)

            if emocion is not None:
                emocion_actual = emocion

                emocion = emocion.lower().strip()

                if emocion in colores_emociones:
                    color_sable = colores_emociones[emocion]

                print("Emocion detectada:", emocion)

            ultima_prediccion = tiempo_actual

        alto, ancho = frame.shape[:2]

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = max(
            int(time.monotonic() * 1000),
            ultimo_timestamp + 1
        )

        ultimo_timestamp = timestamp_ms

        # Detectar las manos.
        resultados = hand_landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        manos = resultados.hand_landmarks

        # --------------------------------------------------
        # CALCULAR POSICION DE LAS MANOS
        # --------------------------------------------------

        posiciones = []
        tamanos = []

        for mano in manos:

            # Muñeca y nudillo del dedo corazón.
            muneca = mano[0]
            nudillo = mano[9]

            punto_muneca = (
                int(muneca.x * ancho),
                int(muneca.y * alto)
            )

            punto_nudillo = (
                int(nudillo.x * ancho),
                int(nudillo.y * alto)
            )

            # Centro aproximado de la palma.
            centro = punto_nudillo

            # Tamaño aproximado de la mano.
            tamano = np.linalg.norm(
                np.array(punto_nudillo) -
                np.array(punto_muneca)
            )

            posiciones.append(centro)
            tamanos.append(tamano)

        # --------------------------------------------------
        # SELECCIONAR LA MANO QUE SUJETA LA ESPADA
        # --------------------------------------------------

        punto_espada = None
        tamano_mano = None
        angulo = ultimo_angulo

        if len(posiciones) > 0:

            # En el primer frame utilizamos la primera mano.
            # Después intentamos mantener la misma mediante
            # la proximidad a su última posición.
            if ultima_posicion_mano is None:

                indice_mano = 0

            else:

                distancias = [
                    np.linalg.norm(
                        np.array(posicion) -
                        np.array(ultima_posicion_mano)
                    )
                    for posicion in posiciones
                ]

                indice_mano = int(np.argmin(distancias))

            punto_espada = posiciones[indice_mano]
            tamano_mano = tamanos[indice_mano]

            # Guardar la posición para el siguiente frame.
            ultima_posicion_mano = punto_espada

            mano = manos[indice_mano]

            # Obtener los nudillos del índice y del meñique.
            indice = mano[5]
            menique = mano[17]

            # Calcular el vector tridimensional.
            dx = (indice.x - menique.x) * ancho
            dy = (indice.y - menique.y) * alto
            dz = (indice.z - menique.z) * ancho

            # Magnitud del vector 3D.
            magnitud = np.sqrt(dx**2 + dy**2 + dz**2)

            if magnitud > 0.00001:

                # Calcular la orientación 2D para la empuñadura.
                distancia_2d = np.sqrt(dx**2 + dy**2)

                if distancia_2d > 15:

                    angulo = np.degrees(
                        np.arctan2(dy, dx)
                    )

                    ultimo_angulo = angulo

                # Proyectar el vector 3D sobre la imagen.
                # No normalizamos esta proyección para conservar
                # el efecto de profundidad.
                direccion = np.array([
                    dx / magnitud,
                    dy / magnitud
                ])

            else:

                # Si el vector no es válido, conservar la última
                # orientación y utilizar una dirección 2D.
                direccion = np.array([
                    np.cos(np.radians(ultimo_angulo)),
                    np.sin(np.radians(ultimo_angulo))
                ])

        else:

            ultima_posicion_mano = None

        
        # --------------------------------------------------
        # DIBUJAR LA ESPADA
        # --------------------------------------------------

        if punto_espada is not None:

            # Dibujar la empuñadura utilizando la misma dirección
            # tridimensional proyectada que utiliza la hoja.
            frame = dibujar_empunadura(
                frame,
                punto_espada,
                direccion,
                tamano_mano
            )

            # La hoja comienza cerca del emisor.
            inicio = (
                int(punto_espada[0] + direccion[0] * tamano_mano * 0.8),
                int(punto_espada[1] + direccion[1] * tamano_mano * 0.8)
            )

            # Longitud de la hoja.
            fin = (
                int(punto_espada[0] + direccion[0] * tamano_mano * 6),
                int(punto_espada[1] + direccion[1] * tamano_mano * 6)
            )

            frame = dibujar_hoja(
                frame,
                inicio,
                fin,
                color_sable
            )


        # --------------------------------------------------
        # MOSTRAR RESULTADO
        # --------------------------------------------------

        cv2.imshow("Espada laser", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break


# --------------------------------------------------
# LIBERAR RECURSOS
# --------------------------------------------------

camara.release()
hand_landmarker.close()
cv2.destroyAllWindows()

Modelo de emociones cargado.
Clases: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: neutral
Emocion detectada: happy
Emocion detectada: happy
Emocion detectada: happy
Emocion detectada: happy
Emocion detectada: happy
Emocion detectada: neutral
Emocion detectada: fearful
Emocion detectada: angry
Emocion detectada: angry
Emocion detectada: angry
Emocion detectada: angry
Emocion detectada: angry
Emocion detectada: angry
Emocion detectada: angry
Emocion detectada: sad
Emocion detectada: sad
Emocion detectada: fearful
Emocion detectada: sad
Emocion detectada: neutral
Emocion